# Loading and displaying the data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("faribatasniakhan/bangla-spam-sms")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/bangla-spam-sms


In [ ]:
import pandas as pd

sms_spam = pd.read_csv('/kaggle/input/bangla-spam-sms/bangla spam2.csv', sep=',', header=None)
# Update with the actual filename if it is different.
header = ['Label', 'SMS']
sms_spam.columns = header
print(sms_spam.shape)
sms_spam.sample(5)

(504, 2)


,Label,SMS
103,ham,প্রমোশনাল এসমএস না চাইলে *৭# ডায়াল করে চালু কর...
342,ham,এভাবে কি স্মার্ট বাংলাদেশের স্বপ্ন দেখা সম্ভব
435,ham,আমাদের সাথে হেড অফিসে সফটওয়্যার ডিপার্টমেন্ট ব...
430,ham,ওই শ্রাবন্তী কই থাকে কি করে কেউ জানস?!
291,ham,আমার নামটা বাচ্চাদের মুখস্ত করার জন্য কঠিন।ওরে...


# Proportion of spam and ham messages in the dataset.

In [ ]:
sms_spam['Label'].value_counts(normalize=True)

,proportion
Label,
ham,0.569444
spam,0.430556


# Shuffle and split data

In [ ]:
# Randomize the dataset
data_randomized = sms_spam.sample(frac=1, random_state=1)

# Calculate index for split
training_test_index = round(len(data_randomized) * 0.8)

# Split into training and test sets
training_set = data_randomized[:training_test_index].reset_index(drop=True)
test_set = data_randomized[training_test_index:].reset_index(drop=True)

print(training_set.shape)
print(test_set.shape)

(403, 2)
(101, 2)


# Proportion of spam and ham messages in the training dataset

In [ ]:
training_set['Label'].value_counts(normalize=True)

,proportion
Label,
ham,0.568238
spam,0.431762


# Proportion of spam and ham messages in the test dataset

In [ ]:
test_set['Label'].value_counts(normalize=True)

,proportion
Label,
ham,0.574257
spam,0.425743


# Clean the messages

**Remove extra whitespace:**

In [ ]:
training_set['SMS'] = training_set['SMS'].str.replace('\s+', ' ')


**Stopword removal**

In [ ]:
import re

# Sample Bangla stopwords (you can expand this list)
bangla_stopwords = [
    'অতএব', 'অথচ', 'অথবা', 'অনুযায়ী', 'অনেক', 'অনেকে', 'অনেকেই', 'অন্তত',
    'অন্য', 'অবধি', 'অবশ্য', 'অর্থাত', 'আই', 'আগামী', 'আগে', 'আগেই', 'আছে',
    'আজ', 'আদ্যভাগে', 'আপনার', 'আপনি', 'আবার', 'আমরা', 'আমাকে', 'আমাদের',
    'আমার', 'আমি', 'আর', 'আরও', 'ই', 'ইত্যাদি', 'ইহা', 'উচিত', 'উত্তর', 'উনি',
    'উপর', 'উপরে', 'এ', 'এঁদের', 'এঁরা', 'এই', 'একই', 'একটি', 'একবার', 'একে',
    'এক্', 'এখন', 'এখনও', 'এখানে', 'এখানেই', 'এটা', 'এটাই', 'এটি', 'এত',
    'এতটাই', 'এতে', 'এদের', 'এব', 'এবং', 'এবার', 'এমন', 'এমনকী', 'এমনি', 'এর',
    'এরা', 'এল', 'এস', 'এসে', 'ঐ', 'ও', 'ওঁদের', 'ওঁর', 'ওঁরা', 'ওই', 'ওকে',
    'ওখানে', 'ওদের', 'ওর', 'ওরা', 'কখনও', 'কত', 'কবে', 'কমনে', 'কয়েক',
    'কয়েকটি', 'করছে', 'করছেন', 'করতে', 'করবে', 'করবেন', 'করলে', 'করলেন',
    'করা', 'করাই', 'করায়', 'করার', 'করি', 'করিতে', 'করিয়া', 'করিয়ে', 'করে',
    'করেই', 'করেছিলেন', 'করেছে', 'করেছেন', 'করেন', 'কাউকে', 'কাছ', 'কাছে',
    'কাজ', 'কাজে', 'কারও', 'কারণ', 'কি', 'কিংবা', 'কিছু', 'কিছুই', 'কিন্তু',
    'কী', 'কে', 'কেউ', 'কেউই', 'কেখা', 'কেন', 'কোটি', 'কোন', 'কোনও', 'কোনো',
    'ক্ষেত্রে', 'কয়েক', 'খুব', 'গিয়ে', 'গিয়েছে',
    'গুলি', 'গেছে', 'গেল', 'গেলে', 'গোটা', 'চলে', 'চান', 'চায়', 'চার', 'চালু',
    'চেয়ে', 'চেষ্টা', 'ছাড়া', 'ছাড়াও', 'ছিল', 'ছিলেন', 'জন', 'জনকে', 'জনের',
    'জন্য', 'জন্যওজে', 'জানতে', 'জানা', 'জানানো', 'জানায়', 'জানিয়ে',
    'জানিয়েছে', 'জে', 'জ্নজন', 'টি', 'ঠিক', 'তখন', 'তত', 'তথা', 'তবু', 'তবে',
    'তা', 'তাঁকে', 'তাঁদের', 'তাঁর', 'তাঁরা', 'তাঁাহারা', 'তাই', 'তাও', 'তাকে',
    'তাতে', 'তাদের', 'তার', 'তারপর', 'তারা', 'তারৈ', 'তাহলে', 'তাহা', 'তাহাতে',
    'তাহার', 'তিনঐ', 'তিনি', 'তিনিও', 'তুমি', 'তুলে', 'তেমন', 'তো', 'তোমার',
    'থাকবে', 'থাকবেন', 'থাকা', 'থাকায়', 'থাকে', 'থাকেন', 'থেকে', 'থেকেই',
    'থেকেও', 'দিকে', 'দিতে', 'দিন', 'দিয়ে', 'দিয়েছে', 'দিয়েছেন', 'দিলেন',
    'দু', 'দুই', 'দুটি', 'দুটো', 'দেওয়া', 'দেওয়ার',
    'দেখতে', 'দেখা', 'দেখে', 'দেন', 'দেয়', 'দ্বারা', 'ধরা', 'ধরে', 'ধামার',
    'নতুন', 'নয়', 'না', 'নাই', 'নাকি', 'নাগাদ', 'নানা', 'নিজে', 'নিজেই',
    'নিজেদের', 'নিজের', 'নিতে', 'নিয়ে',
    'নেই', 'নেওয়া', 'নেওয়ার',
    'পক্ষে', 'পর', 'পরে', 'পরেই', 'পরেও', 'পর্যন্ত', 'পাওয়া', 'পাচ', 'পারি',
    'পারে', 'পারেন', 'পি', 'পেয়ে', 'পেয়্র্', 'প্রতি', 'প্রথম', 'প্রভৃতি',
    'প্রযন্ত', 'প্রাথমিক', 'প্রায়',
    'ফলে', 'ফিরে', 'ফের', 'বক্তব্য', 'বদলে', 'বন', 'বরং', 'বলতে', 'বলল',
    'বললেন', 'বলা', 'বলে', 'বলেছেন', 'বলেন', 'বসে', 'বহু', 'বা', 'বাদে', 'বার',
    'বি', 'বিনা', 'বিভিন্ন', 'বিশেষ', 'বিষয়টি', 'বেশ', 'বেশি', 'ব্যবহার',
    'ব্যাপারে', 'ভাবে', 'ভাবেই', 'মতো', 'মতোই', 'মধ্যভাগে', 'মধ্যে', 'মধ্যেই',
    'মধ্যেও', 'মনে', 'মাত্র', 'মাধ্যমে', 'মোট', 'মোটেই', 'যখন', 'যত', 'যতটা',
    'যথেষ্ট', 'যদি', 'যদিও', 'যা', 'যাঁর', 'যাঁরা', 'যাওয়া', 'যাওয়ার',
    'যাকে', 'যাচ্ছে', 'যাতে', 'যাদের', 'যান', 'যাবে', 'যায়', 'যার', 'যারা',
    'যিনি', 'যে', 'যেখানে', 'যেতে', 'যেন', 'যেমন', 'র', 'রকম', 'রয়েছে', 'রাখা',
    'রেখে', 'লক্ষ', 'শুধু', 'শুরু', 'সঙ্গে', 'সঙ্গেও', 'সব', 'সবার', 'সমস্ত',
    'সম্প্রতি', 'সহ', 'সহিত', 'সাধারণ', 'সামনে', 'সি', 'সুতরাং', 'সে', 'সেই',
    'সেখান', 'সেখানে', 'সেটা', 'সেটাই', 'সেটাও', 'সেটি', 'স্পষ্ট', 'স্বয়ং',
    'হইতে', 'হইবে', 'হইয়া', 'হওয়া', 'হওয়ায়', 'হওয়ার',
    'হচ্ছে', 'হত', 'হতে', 'হতেই', 'হন', 'হবে', 'হবেন', 'হয়', 'হয়তো', 'হয়নি',
    'হয়ে', 'হয়েই', 'হয়েছিল', 'হয়েছে', 'হয়েছেন', 'হল', 'হলে', 'হলেই', 'হলেও',
    'হলো', 'হাজার', 'হিসাবে', 'হৈলে', 'হোক',
]

def clean_bangla_text(text):
    text = text.lower()
    text = re.sub(r'[^\u0980-\u09FF\s]', ' ', text)  # Keep only Bangla characters and spaces
    words = text.split()
    words = [word for word in words if word not in bangla_stopwords]  # remove stopwords
    return ' '.join(words)

training_set['SMS'] = training_set['SMS'].apply(clean_bangla_text)


In [ ]:
test_set['SMS'] = test_set['SMS'].apply(clean_bangla_text)

In [ ]:
training_set.head(20)

,Label,SMS
0,spam,অফারেই কিনে নিন পছন্দের পণ্য
1,ham,আল্লাহ ভাই সকল মুদ্দারকে জান্নাতবাসী আমিন
2,spam,ঘরে রোজার বাজার ডিস্কাউন্টকোড গিফটকোড
3,ham,প্রিয় গ্রাহক নম্বরে ৮৫০ ০ টাকা লোকাল বিল প্রদ...
4,ham,জলদি আসো
5,spam,খেলাধুলার সর্বশেষ খবর পেতে ভিজিট করুন ট্যাক্সস...
6,ham,পাওয়া ২ কার্ল্প্রিটকে পরেরবার হইলে ওরে সরি বলব...
7,spam,কুরআন পাকের ১৪৪১ বছর হয়ে যাক কুরআন পাকের মুবার...
8,spam,অভিনন্দন বিকাশের নিয়মিত ব্যবহারকারী হিসেবে বিক...
9,spam,ক্লিক করুনঃ বিস্তারিত কল করুনঃ ১৬৪৬০ শর্ত প্রয...


# Build the vocabulary

In [ ]:
training_set['SMS'] = training_set['SMS'].str.split()

vocabulary = []
for sms in training_set['SMS']:
   for word in sms:
      vocabulary.append(word)

vocabulary = list(set(vocabulary))

In [ ]:
len(vocabulary)

2120

# Count word frequencies

In [ ]:
word_counts_per_sms = {unique_word: [0] * len(training_set['SMS']) for unique_word in vocabulary}

for index, sms in enumerate(training_set['SMS']):
   for word in sms:
      word_counts_per_sms[word][index] += 1

In [ ]:
word_counts = pd.DataFrame(word_counts_per_sms)
word_counts.sample(10)

,কাজের,সম্পূর্ণ,টিপস,মনিরুল,মিঃ,লাইফের,মাল্টা,৪৫,২৯৯,কেমন,...,মাল্টা৳২২৯,পড়সে,ায়,হইল,৪০,জিতি,মেডিকেল,মানুষের,রইল,বাড়িতে
100,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
314,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
171,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
121,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
209,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
156,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
221,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
346,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
90,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
309,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Merge word counts with original data

In [ ]:
training_set_clean = pd.concat([training_set, word_counts], axis=1)
training_set_clean.head()

,Label,SMS,কাজের,সম্পূর্ণ,টিপস,মনিরুল,মিঃ,লাইফের,মাল্টা,৪৫,...,মাল্টা৳২২৯,পড়সে,ায়,হইল,৪০,জিতি,মেডিকেল,মানুষের,রইল,বাড়িতে
0,spam,"[অফারেই, কিনে, নিন, পছন্দের, পণ্য]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,ham,"[আল্লাহ, ভাই, সকল, মুদ্দারকে, জান্নাতবাসী, আমিন]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,spam,"[ঘরে, রোজার, বাজার, ডিস্কাউন্টকোড, গিফটকোড]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,ham,"[প্রিয়, গ্রাহক, নম্বরে, ৮৫০, ০, টাকা, লোকাল, ...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ham,"[জলদি, আসো]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Isolating spam and ham messages first
spam_messages = training_set_clean[training_set_clean['Label'] == 'spam']
ham_messages = training_set_clean[training_set_clean['Label'] == 'ham']

# Calculate probabilities of Spam & Ham
p_spam = len(spam_messages) / len(training_set_clean)
p_ham = len(ham_messages) / len(training_set_clean)


# Total word counts

In [ ]:
# N_Spam
n_words_per_spam_message = spam_messages['SMS'].apply(len)
n_spam = n_words_per_spam_message.sum()

# N_Ham
n_words_per_ham_message = ham_messages['SMS'].apply(len)
n_ham = n_words_per_ham_message.sum()

# N_Vocabulary
n_vocabulary = len(vocabulary)

# Laplace smoothing
alpha = 1

# Calculate word probabilities (Naive Bayes core)

In [ ]:
# Initiate parameters
parameters_spam = {unique_word:0 for unique_word in vocabulary}
parameters_ham = {unique_word:0 for unique_word in vocabulary}

# Calculate parameters
for word in vocabulary:
   n_word_given_spam = spam_messages[word].sum() # spam_messages already defined
   p_word_given_spam = (n_word_given_spam + alpha) / (n_spam + alpha*n_vocabulary)
   parameters_spam[word] = p_word_given_spam

   n_word_given_ham = ham_messages[word].sum() # ham_messages already defined
   p_word_given_ham = (n_word_given_ham + alpha) / (n_ham + alpha*n_vocabulary)
   parameters_ham[word] = p_word_given_ham

# Print-friendly version for testing

In [ ]:
import re

def classify(message):
   '''
   message: a string
   '''
   # Remove non-alphabetical characters (keeping Bengali characters)
   message = re.sub('[^\w\s\u0980-\u09FF]', ' ', message)
   message = message.lower().split()

   p_spam_given_message = p_spam
   p_ham_given_message = p_ham

   for word in message:
      if word in parameters_spam:
         p_spam_given_message *= parameters_spam[word]

      if word in parameters_ham:
         p_ham_given_message *= parameters_ham[word]

   print('P(Spam|message):', p_spam_given_message)
   print('P(Ham|message):', p_ham_given_message)

   if p_ham_given_message > p_spam_given_message:
      print('Label: Ham')
   elif p_ham_given_message < p_spam_given_message:
      print('Label: Spam')
   else:
      print('Equal probabilities, have a human classify this!')

In [ ]:
classify('dada,আমি ফাহিম বলছি । আপনি কেমন আছেন ?')

P(Spam|message): 4.68546200894534e-08
P(Ham|message): 2.015413983838771e-07
Label: Ham


In [ ]:
classify('আপিন three hundred টাকা েপেয়েছন েমেসজ ওপশন এ িগেয় িলখুন START NMZRR send 21213')

P(Spam|message): 1.2182201223257883e-06
P(Ham|message): 7.725753604715288e-07
Label: Spam


In [ ]:
classify('প্রিয়, কানাডাতে IELTS ছাড়া অ্যাপ্লাই করার ২৬ টা স্টাডি স্লট থেকে আপনার স্লটে অ্যাডমিশন পেতে আজকে 3pm-5pm এর মধ্যে আমাদের অফিস ভিসিট করুন।')

P(Spam|message): 2.1455081899906935e-44
P(Ham|message): 9.080790820506161e-47
Label: Spam


In [ ]:
classify('Pureit RO কৰ্েয় ৩০০০ টাকা পযর্ন্ত ছাড়! এছাড়াও থাকেছ না িকেনই ১৫ িদেনর িফৰ্ টৰ্ায়াল অফার! কলকরুন ১৬৬২৭ এ। শতর্ পৰ্েযাজয্**')

P(Spam|message): 1.8831816040339164e-23
P(Ham|message): 6.383348970527159e-28
Label: Spam


In [ ]:
classify("আসসালামু আলাইকুম স্যার। রেজিষ্ট্রেশন লিংক তো কাজ করে না। আমি স্যার বিকাল থেকে চেষ্টা করেছি। রাজ স্যার বললো, সন্ধ্যার নাকি রেজিষ্ট্রেশন শেষ হয়ে গেসে। আমি তো ইমেলটাই পেয়েছি বিকালে দিকে। এখন কি করতে পারি  স্যার। জানালে উপকৃত হবো। ধন্যবাদ স্যার।")

P(Spam|message): 8.773470586016579e-52
P(Ham|message): 2.397833499155972e-48
Label: Ham


In [ ]:
def classify_test_set(message):
   '''
   message: a string
   '''

   message = re.sub('[^\w\s\u0980-\u09FF]', ' ', message)
   message = message.lower().split()

   p_spam_given_message = p_spam
   p_ham_given_message = p_ham

   for word in message:
      if word in parameters_spam:
         p_spam_given_message *= parameters_spam[word]

      if word in parameters_ham:
         p_ham_given_message *= parameters_ham[word]

   if p_ham_given_message > p_spam_given_message:
      return 'ham'
   elif p_spam_given_message > p_ham_given_message:
      return 'spam'
   else:
      return 'needs human classification'

In [ ]:
test_set['predicted'] = test_set['SMS'].apply(classify_test_set)
test_set.sample(100)

,Label,SMS,predicted
64,ham,সোমবার তানির বাসায় যাবো জাকু মারজু প্রমিকেও বলবো,ham
54,ham,আপনাদের যেটা সুবিধা লাগে পড়েন,ham
40,spam,মজার হাসির জোকস পেতে ডায়াল করুন,spam
19,spam,চিরসবুজ বাংলা গান শুনুন এখনই রে কয়েন রিচার্জ,spam
14,ham,পার্সেল আজকে দিবে তুই বিকাশ আজকে পারবি নাহয় কা...,ham
...,...,...,...
31,ham,১০ তারিখ সকালে রওনা দিব শান্তিনগরে ইস্টার্ন প্...,ham
63,spam,আকর্ষণীয় ১৫ ছাড় ৳১০০ উপায় অ্যাপ দিয়ে বিডিটিকেট...,spam
12,spam,সেন্ট্রাল প্লাজা,ham
30,spam,সাথেই থাকুন ডায়েল পেয়ে টাকা ব্যালেন্স ভ্যাট সা...,spam


In [ ]:
correct = 0
total = test_set.shape[0]

for row in test_set.iterrows():
   row = row[1]
   if row['Label'] == row['predicted']:
      correct += 1

print('Correct:', correct)
print('Incorrect:', total - correct)
print('Accuracy:', correct/total)

Correct: 92
Incorrect: 9
Accuracy: 0.9108910891089109
